# Milestone 2 — Member 2: phát hiện LaTeX root

Notebook triển khai **Task 1: `find_root()`** và **Task 2: `gather()`** cho source của **một version**.
Task 3: `build_tree()` sẽ được bổ sung sau.

**Chạy trên Colab:** dùng runtime CPU, chạy các cell từ trên xuống. Cell đầu tiên
kết nối Google Drive; cell cấu hình bên dưới chọn paper và version cần thử.
Không cần clone repo hay cài thêm package để chạy parser.

Dữ liệu nhóm nằm trong `arxiv-project/data/raw/<paper_id>/<version>/`.
Ví dụ đã quan sát: `2609-37833/v1/` chứa `main.tex` và `references.bib`.
Notebook chỉ đọc dữ liệu, không chỉnh sửa source M1 và không chạy toàn bộ corpus.

## 1. Kết nối dữ liệu

Nếu `arxiv-project` đang ở **Được chia sẻ với tôi**, thêm lối tắt của thư mục
vào **Drive của tôi** trước. Đăng nhập đúng tài khoản có quyền truy cập.
Nếu lối tắt có tên hoặc vị trí khác, sửa `RAW_DIR` trong cell cấu hình.
Khi chạy local, cell mount sẽ được bỏ qua; hãy đặt `RAW_DIR` thành đường dẫn local.

In [ ]:
try:
    from google.colab import drive
except ImportError:
    print("Đang chạy ngoài Colab: bỏ qua mount Drive, cấu hình RAW_DIR local ở cell sau.")
else:
    drive.mount("/content/drive")

## 2. Chọn một version

Chỉ cần chỉnh ba giá trị dưới đây. `source_dir` phải trỏ trực tiếp vào thư mục
version chứa `.tex`, không trỏ vào toàn bộ `raw` hoặc paper chứa nhiều version.
Nếu không tìm thấy đường dẫn, dùng bảng **Files** của Colab để lấy **Copy path**.

In [ ]:
from pathlib import Path

RAW_DIR = Path("/content/drive/MyDrive/arxiv-project/data/raw")
PAPER_ID = "2609-37833"
VERSION = "v1"

source_dir = RAW_DIR / PAPER_ID / VERSION

if not source_dir.is_dir():
    raise FileNotFoundError(
        f"Không tìm thấy thư mục version: {source_dir}\n"
        "Kiểm tra mount Drive, lối tắt arxiv-project và giá trị RAW_DIR."
    )

print("Source directory:", source_dir)
print("\nNội dung trực tiếp (tối đa 30 mục):")
entries = sorted(source_dir.iterdir(), key=lambda path: path.name)
for path in entries[:30]:
    kind = "DIR" if path.is_dir() else "FILE"
    print(f"[{kind}] {path.name}")

tex_files = sorted(
    path for path in source_dir.rglob("*")
    if path.is_file() and path.suffix.lower() == ".tex"
)
print("\nSố file .tex:", len(tex_files))
for path in tex_files[:20]:
    print("-", path.relative_to(source_dir).as_posix())
if not tex_files:
    print("Chưa thấy .tex: kiểm tra thư mục, file nén hoặc version không có TeX.")

## 3. Hàm hỗ trợ

Mask comment và các môi trường verbatim phổ biến; đọc file với encoding fallback;
dựng graph `input/include` sơ bộ để đánh giá ứng viên root. Cell này chỉ định nghĩa hàm.

In [ ]:
from pathlib import Path
import re


_VERBATIM_ENVS = {
    "verbatim",
    "verbatim*",
    "Verbatim",
    "lstlisting",
    "minted",
    "comment",
}

_DOCUMENTCLASS_RE = re.compile(
    r"\\documentclass\b"
    r"\s*(?:\[[^\]]*\]\s*)?"
    r"\{([^{}]+)\}",
    flags=re.DOTALL,
)

_BEGIN_DOCUMENT_RE = re.compile(
    r"\\begin\s*\{\s*document\s*\}"
)

_END_DOCUMENT_RE = re.compile(
    r"\\end\s*\{\s*document\s*\}"
)

_INCLUDE_RE = re.compile(
    r"\\(?:input|include)\b"
    r"\s*(?:\{([^{}]*)\}|([^\s%{}]+))"
)


def _mask_non_code(text):
    chars = list(text)

    def blank(start, end):
        for pos in range(start, end):
            if chars[pos] not in "\r\n":
                chars[pos] = " "

    i = 0
    n = len(text)

    while i < n:
        if text[i] == "%":
            backslashes = 0
            j = i - 1

            while j >= 0 and text[j] == "\\":
                backslashes += 1
                j -= 1

            if backslashes % 2 == 0:
                end = text.find("\n", i)

                if end == -1:
                    end = n

                blank(i, end)
                i = end
                continue

        if text[i] == "\\":
            if i + 1 < n and not text[i + 1].isalpha():
                i += 2
                continue

            verb_match = re.match(r"\\verb\*?(?![A-Za-z])", text[i:])

            if verb_match:
                delimiter_pos = i + verb_match.end()

                if (
                    delimiter_pos < n
                    and not text[delimiter_pos].isspace()
                ):
                    delimiter = text[delimiter_pos]
                    end = text.find(delimiter, delimiter_pos + 1)
                    newline = text.find("\n", delimiter_pos + 1)

                    if end != -1 and (newline == -1 or end < newline):
                        blank(i, end + 1)
                        i = end + 1
                        continue

            # Bỏ qua các môi trường chứa literal text.
            env_match = re.match(
                r"\\begin\s*\{([^{}]+)\}",
                text[i:],
            )

            if env_match and env_match.group(1) in _VERBATIM_ENVS:
                env_name = env_match.group(1)
                end_pattern = re.compile(
                    r"\\end\s*\{" + re.escape(env_name) + r"\}"
                )

                end_match = end_pattern.search(
                    text,
                    i + env_match.end(),
                )

                end = end_match.end() if end_match else n
                blank(i, end)
                i = end
                continue

            i += 1

            while i < n and text[i].isalpha():
                i += 1

            continue

        i += 1

    return "".join(chars)


def _read_tex(path):
    raw = path.read_bytes()

    try:
        return raw.decode("utf-8-sig"), None
    except UnicodeDecodeError:
        return (
            raw.decode("latin-1"),
            "UTF-8 decoding failed; used Latin-1 fallback.",
        )


def _resolve_include(argument, current_file, source_dir, known_files):
    argument = argument.strip()

    if not argument or any(
        token in argument for token in ("\\", "#", "{", "}", "\x00")
    ):
        return None

    relative_path = Path(argument)

    if relative_path.is_absolute() or relative_path.drive:
        return None

    if not relative_path.suffix:
        relative_path = relative_path.with_suffix(".tex")

    for base_dir in (current_file.parent, source_dir):
        candidate = (base_dir / relative_path).resolve()

        if (
            candidate.is_relative_to(source_dir)
            and candidate in known_files
        ):
            return candidate

    return None


def _reachable_files(start, graph):
    visited = set()
    pending = [start]

    while pending:
        current = pending.pop()

        if current in visited:
            continue

        visited.add(current)
        pending.extend(graph[current] - visited)

    return visited

## 4. Tìm root

Ứng viên phải có `documentclass` ngoài comment/verbatim. Class `standalone` bị loại.
Khi có nhiều ứng viên, heuristic xét dấu hiệu document, quan hệ include, tên main/ms
và tên tài liệu phụ. `ambiguous=True` yêu cầu kiểm tra thủ công.

**Giới hạn:** chưa diễn giải điều kiện TeX, macro trong đường dẫn và `includeonly`.
Điểm số không phải xác suất. `status="ok"` chỉ nghĩa là đã chọn được ứng viên.

In [ ]:
def find_root(source_dir):
    source_dir = Path(source_dir).expanduser().resolve()

    if not source_dir.is_dir():
        raise NotADirectoryError(
            f"Không tìm thấy thư mục source: {source_dir}"
        )

    def relative_name(path):
        return path.relative_to(source_dir).as_posix()

    result = {
        "status": "no_root",
        "source_dir": str(source_dir),
        "root_path": None,
        "selection_reason": [],
        "ambiguous": False,
        "candidates": [],
        "tex_file_count": 0,
        "warnings": [],
    }

    tex_files = sorted({
        path.resolve()
        for path in source_dir.rglob("*")
        if (
            path.is_file()
            and path.suffix.lower() == ".tex"
            and path.resolve().is_relative_to(source_dir)
        )
    })

    result["tex_file_count"] = len(tex_files)

    if not tex_files:
        result["warnings"].append("Không tìm thấy file .tex.")
        return result

    clean_sources = {}

    for path in tex_files:
        try:
            text, encoding_warning = _read_tex(path)
        except OSError as exc:
            result["warnings"].append(
                f"Không đọc được {relative_name(path)}: {exc}"
            )
            continue

        clean_sources[path] = _mask_non_code(text)

        if encoding_warning:
            result["warnings"].append(
                f"{relative_name(path)}: {encoding_warning}"
            )

    known_files = set(clean_sources)

    graph = {path: set() for path in known_files}
    incoming = {path: set() for path in known_files}

    for path, text in clean_sources.items():
        for match in _INCLUDE_RE.finditer(text):
            argument = (
                match.group(1)
                if match.group(1) is not None
                else match.group(2)
            )

            target = _resolve_include(
                argument,
                current_file=path,
                source_dir=source_dir,
                known_files=known_files,
            )

            if target is not None:
                graph[path].add(target)
                incoming[target].add(path)

    candidates = []

    for path, text in clean_sources.items():
        declarations = list(_DOCUMENTCLASS_RE.finditer(text))

        if not declarations:
            continue

        class_name = declarations[0].group(1).strip()
        excluded = class_name.casefold() == "standalone"

        reachable = _reachable_files(path, graph)

        has_begin_here = bool(_BEGIN_DOCUMENT_RE.search(text))
        has_begin_reachable = any(
            _BEGIN_DOCUMENT_RE.search(clean_sources[file])
            for file in reachable
        )
        has_end_reachable = any(
            _END_DOCUMENT_RE.search(clean_sources[file])
            for file in reachable
        )

        score = 0
        reasons = []

        if has_begin_here:
            score += 40
            reasons.append("+40: có begin{document} trong chính file")
        elif has_begin_reachable:
            score += 25
            reasons.append("+25: begin{document} nằm trong file reachable")
        else:
            reasons.append("Không tìm thấy begin{document} reachable")

        if has_end_reachable:
            score += 5
            reasons.append("+5: có end{document} reachable")

        other_parents = incoming[path] - {path}

        if not other_parents:
            score += 20
            reasons.append("+20: không bị file .tex khác include")

        stem = path.stem.casefold()

        if stem in {"main", "ms"}:
            score += 20
            reasons.append("+20: tên file main/ms")

        name_tokens = set(re.split(r"[^a-z0-9]+", stem))

        auxiliary_tokens = {
            "rebuttal",
            "response",
            "supp",
            "supplement",
            "supplementary",
            "standalone",
        }

        if name_tokens & auxiliary_tokens:
            score -= 40
            reasons.append("-40: tên file có dấu hiệu tài liệu phụ")

        reachable_children = len(reachable) - 1
        include_bonus = min(reachable_children, 5)

        if include_bonus:
            score += include_bonus
            reasons.append(
                f"+{include_bonus}: có file .tex reachable"
            )

        if excluded:
            reasons.append("Loại khỏi lựa chọn: class standalone")

        if len(declarations) > 1:
            result["warnings"].append(
                f"{relative_name(path)} có nhiều documentclass; "
                "cần kiểm tra thủ công."
            )

        candidates.append({
            "path": relative_name(path),
            "documentclass": class_name,
            "excluded": excluded,
            "score": score,
            "has_begin_document": has_begin_here,
            "has_reachable_begin_document": has_begin_reachable,
            "reachable_tex_count": len(reachable),
            "included_by": sorted(
                relative_name(parent)
                for parent in other_parents
            ),
            "reasons": reasons,
        })

    candidates.sort(
        key=lambda item: (
            item["excluded"],
            -item["score"],
            item["path"],
        )
    )

    result["candidates"] = candidates

    eligible = [
        candidate
        for candidate in candidates
        if not candidate["excluded"]
    ]

    if not eligible:
        result["warnings"].append(
            "Không có ứng viên documentclass hợp lệ "
            "sau khi loại standalone."
        )
        return result

    selected = eligible[0]

    result["status"] = "ok"
    result["root_path"] = selected["path"]
    result["selection_reason"] = selected["reasons"]

    if len(eligible) > 1:
        result["warnings"].append(
            f"Có {len(eligible)} ứng viên root; "
            "đã chọn bằng heuristic."
        )

        score_gap = selected["score"] - eligible[1]["score"]

        if score_gap < 15:
            result["ambiguous"] = True
            result["warnings"].append(
                "Các ứng viên dẫn đầu có điểm gần nhau; "
                "cần kiểm tra thủ công."
            )

    if not selected["has_reachable_begin_document"]:
        result["warnings"].append(
            "Root được chọn không dẫn tới begin{document} "
            "mà bộ quét hiện tại nhận diện được."
        )

    return result

## 5. Chạy và xem ứng viên

Với version trong ảnh, kỳ vọng chọn `main.tex` nếu khai báo documentclass hợp lệ.
Bảng bên dưới giữ cả ứng viên bị loại để tiện kiểm tra.

In [ ]:
from IPython.display import display

root_result = find_root(source_dir)

print("Paper:", PAPER_ID, "| Version:", VERSION)
print("Status:", root_result["status"])
print("Root:", root_result["root_path"])
print("Ambiguous:", root_result["ambiguous"])
print("Number of .tex files:", root_result["tex_file_count"])

print("\nSelection reasons:")
for reason in root_result["selection_reason"]:
    print("-", reason)

print("\nWarnings:")
if not root_result["warnings"]:
    print("(none)")
for warning in root_result["warnings"]:
    print("-", warning)

rows = [
    {key: item[key] for key in (
        "path", "documentclass", "excluded", "score",
        "has_reachable_begin_document", "reachable_tex_count", "included_by"
    )}
    for item in root_result["candidates"]
]
if rows:
    try:
        import pandas as pd
    except ImportError:
        for row in rows:
            print(row)
    else:
        display(pd.DataFrame(rows))

## 6. Kiểm tra root được chọn

Xem khai báo documentclass và phần đầu source để xác nhận đây là bài chính.
Không có root thì cell này chỉ thông báo, không đọc một đường dẫn rỗng.

In [ ]:
root_file = None

if root_result["status"] != "ok":
    print("Chưa chọn được root. Xem warnings và kiểm tra source của version.")
else:
    root_file = Path(root_result["source_dir"]) / root_result["root_path"]
    root_text, encoding_warning = _read_tex(root_file)
    masked_text = _mask_non_code(root_text)
    declaration = _DOCUMENTCLASS_RE.search(masked_text)

    print("Root file:", root_file)
    print("Documentclass:", declaration.group(0) if declaration else "Không thấy")
    print("begin{document} trong root:", bool(_BEGIN_DOCUMENT_RE.search(masked_text)))
    if encoding_warning:
        print("Encoding warning:", encoding_warning)
    print("\n--- Preview: tối đa 4000 ký tự đầu ---")
    print(root_text[:4000])

## Sau Task 1

Giữ `source_dir`, `root_file` và `root_result` để chạy Task 2 bên dưới.
Nếu root mơ hồ, kiểm tra và chọn lại root trước khi gather.


# Task 2 — Gom source LaTeX nhiều file

Chạy sau Task 1 trong cùng runtime. `gather()` chỉ đọc các file theo input/include
reachable từ root; không nối tất cả file trong thư mục.

- Loại comment `%`, giữ ký tự `\%` và literal text trong verbatim.
- Mở rộng include theo thứ tự, kể cả nhiều tầng; giữ các lần include lặp hợp lệ.
- Chặn vòng lặp bằng active stack, không dùng tập visited để loại lần xuất hiện.
- Cache nội dung đã đọc để giảm truy cập Drive.
- Gộp xong mới tách preamble/body theo begin/end document.
- File thiếu, vòng lặp và đường dẫn macro được giữ nguyên lệnh trong output và báo lỗi.

Chính sách đường dẫn: thử từ thư mục file gọi, sau đó thư mục version; nếu không
có extension thì ưu tiên `.tex`. Nếu có nhiều kết quả, chọn theo thứ tự đó và cảnh báo.
Không cho phép truy cập file nằm ngoài thư mục version.

## 7. Định nghĩa gather

Giới hạn: chưa thực thi macro/điều kiện TeX và includeonly, chưa hỗ trợ subimport/import
hoặc lệnh include tự định nghĩa. Đây là bộ quét lexical, không phải TeX interpreter;
include nằm trong macro chưa gọi cũng có thể bị nhận diện. Với các cấu trúc này,
cần kiểm tra thủ công thay vì coi output là đường biên dịch đã được xác minh.

`line_in_cleaned_source` là dòng sau bỏ comment, không phải source map gốc.
Mặc định giới hạn độ sâu 64, số lần mở rộng 10.000 và văn bản 20 triệu ký tự.

In [ ]:
from pathlib import Path
import re


def _strip_tex_comments(text):
    """Remove real % comments (including their newline), preserving verbatim."""
    pieces = []
    i = 0
    while i < len(text):
        if text[i] == "%":
            end = text.find("\n", i)
            i = len(text) if end == -1 else end + 1
            continue
        if text[i] == "\\":
            # Control symbols, including \%, \\, \{.
            if i + 1 < len(text) and not text[i + 1].isalpha():
                pieces.append(text[i:i + 2])
                i += 2
                continue
            verb = re.match(r"\\verb\*?(?![A-Za-z])", text[i:])
            if verb:
                start = i + verb.end()
                if start < len(text) and not text[start].isspace():
                    end = text.find(text[start], start + 1)
                    newline = text.find("\n", start + 1)
                    if end != -1 and (newline == -1 or end < newline):
                        pieces.append(text[i:end + 1])
                        i = end + 1
                        continue
            env = re.match(r"\\begin\s*\{([^{}]+)\}", text[i:])
            if env and env.group(1) in _VERBATIM_ENVS:
                closing = re.compile(
                    r"\\end\s*\{" + re.escape(env.group(1)) + r"\}"
                ).search(text, i + env.end())
                end = closing.end() if closing else len(text)
                pieces.append(text[i:end])
                i = end
                continue
            command = re.match(r"\\[A-Za-z]+", text[i:])
            if command:
                pieces.append(command.group())
                i += command.end()
                continue
        pieces.append(text[i])
        i += 1
    return "".join(pieces)


def _real_command(text, start):
    """Reject matches beginning at the second slash of an escaped backslash."""
    previous = start - 1
    count = 0
    while previous >= 0 and text[previous] == "\\":
        count += 1
        previous -= 1
    return count % 2 == 0


def gather(source_dir, root_file, *, max_depth=64,
           max_expansions=10000, max_chars=20_000_000):
    """
    Expand input/include recursively for ONE version, without writing files.

    Depends on Task 1 helpers: _read_tex, _mask_non_code, _VERBATIM_ENVS.
    root_file may be an absolute path or a path relative to source_dir.

    status: ok / partial / no_document / error.
    partial output must be reviewed before downstream parsing.
    This is a lexical gatherer, not a TeX interpreter.
    """
    source_dir = Path(source_dir).expanduser().resolve()
    if not source_dir.is_dir():
        raise NotADirectoryError(source_dir)
    if root_file is None:
        raise ValueError("root_file is None; run find_root() successfully first.")
    if min(max_depth, max_expansions, max_chars) <= 0:
        raise ValueError("Gather limits must be positive.")

    root = Path(root_file).expanduser()
    root = (root if root.is_absolute() else source_dir / root).resolve()
    if not root.is_relative_to(source_dir):
        raise ValueError("Root must be inside this version's source directory.")
    if not root.is_file():
        raise FileNotFoundError(root)

    result = {
        "status": "ok", "source_dir": str(source_dir),
        "root_path": root.relative_to(source_dir).as_posix(),
        "expanded_text": "", "preamble": "", "body": "",
        "trailing_text": "", "document_found": False,
        "files_read": [], "include_events": [], "issues": [],
        "warnings": [], "expanded_char_count": 0,
    }
    cache = {}
    expansions = 0

    def name(path):
        return path.relative_to(source_dir).as_posix()

    def issue(code, message, **details):
        result["issues"].append({"code": code, "message": message, **details})
        result["warnings"].append(f"[{code}] {message}")

    def resolve(argument, current):
        argument = argument.strip()
        if not argument or any(c in argument for c in "\\#{}\x00"):
            return [], "unsupported_path"
        path = Path(argument)
        if path.is_absolute() or path.drive:
            return [], "outside_source"
        # Try current file's directory, then version directory. Prefer .tex
        # when the command omitted the extension; never search by basename.
        variants = [path] if path.suffix else [path.with_suffix(".tex"), path]
        matches = []
        outside = False
        for base in (current.parent, source_dir):
            for variant in variants:
                target = (base / variant).resolve()
                if not target.is_relative_to(source_dir):
                    outside = True
                    continue
                if target.is_file() and target not in matches:
                    matches.append(target)
        return matches, "outside_source" if outside else "missing_file"

    class GatherLimitError(RuntimeError):
        pass

    def expand(path, stack):
        nonlocal expansions
        if path not in cache:
            text, encoding_warning = _read_tex(path)
            cache[path] = _strip_tex_comments(text)
            result["files_read"].append(name(path))
            if encoding_warning:
                issue("encoding_fallback", f"{name(path)}: {encoding_warning}")

        text = cache[path]
        masked = _mask_non_code(text)
        # These constructs need TeX evaluation; make the limitation visible.
        unsupported = re.search(
            r"\\(?:includeonly|iffalse|iftrue|ifdefined|ifx|ifnum|ifdim|ifcase)\b",
            masked,
        )
        if unsupported and not any(
            item.get("file") == name(path) and item["code"] == "tex_control"
            for item in result["issues"]
        ):
            issue("tex_control", f"{name(path)} contains unsupported TeX control flow/includeonly.", file=name(path))

        chunks = []
        length = 0

        def append(chunk):
            nonlocal length
            length += len(chunk)
            if length > max_chars:
                raise GatherLimitError(f"Expanded text exceeds max_chars={max_chars}.")
            chunks.append(chunk)

        cursor = 0
        for match in _INCLUDE_RE.finditer(masked):
            if not _real_command(masked, match.start()):
                continue
            append(text[cursor:match.start()])
            original = text[match.start():match.end()]
            argument = match.group(1) if match.group(1) is not None else match.group(2)
            event = {
                "source": name(path), "argument": argument.strip(),
                # Line after comment removal; not an original-source line map.
                "line_in_cleaned_source": text.count("\n", 0, match.start()) + 1,
                "target": None, "status": "pending",
            }
            result["include_events"].append(event)
            targets, failure = resolve(argument, path)
            if not targets:
                event["status"] = failure
                issue(failure, f"{name(path)}: cannot resolve {original!r}.")
                append(original)
            else:
                target = targets[0]
                event["target"] = name(target)
                if len(targets) > 1:
                    event["resolution_candidates"] = [name(p) for p in targets]
                    issue("ambiguous_include", f"{name(path)}: multiple matches for {argument!r}; chose {name(target)}.")
                if target in stack:
                    event["status"] = "cycle"
                    issue("cycle", " -> ".join(name(p) for p in (*stack, target)))
                    append(original)
                elif len(stack) >= max_depth:
                    event["status"] = "depth_limit"
                    issue("depth_limit", f"Cannot expand {name(target)} at max_depth={max_depth}.")
                    append(original)
                else:
                    expansions += 1
                    if expansions > max_expansions:
                        event["status"] = "expansion_limit"
                        raise GatherLimitError(f"Exceeded max_expansions={max_expansions}.")
                    try:
                        content = expand(target, (*stack, target))
                    except OSError as exc:
                        event["status"] = "read_error"
                        issue("read_error", f"{name(target)}: {exc}")
                        append(original)
                    else:
                        event["status"] = "expanded"
                        # A source file ends a line at EOF in ordinary TeX use.
                        append(content + ("\n" if content and not content.endswith("\n") else ""))
            cursor = match.end()
        append(text[cursor:])
        return "".join(chunks)

    try:
        expanded = expand(root, (root,))
    except (OSError, GatherLimitError) as exc:
        issue("gather_error", str(exc))
        for event in result["include_events"]:
            if event["status"] == "pending":
                event["status"] = "aborted"
        result["status"] = "error"
        return result

    result["expanded_text"] = expanded
    result["expanded_char_count"] = len(expanded)
    masked = _mask_non_code(expanded)
    begins = [m for m in _BEGIN_DOCUMENT_RE.finditer(masked) if _real_command(masked, m.start())]
    if not begins:
        result["preamble"] = expanded
        result["status"] = "no_document"
        issue("missing_begin_document", "No begin{document} after gathering.")
        return result

    begin = begins[0]
    result["document_found"] = True
    result["preamble"] = expanded[:begin.start()]
    if len(begins) > 1:
        issue("multiple_documents", "Multiple begin{document} markers; inspect compilation path.")
    ends = [m for m in _END_DOCUMENT_RE.finditer(masked, begin.end()) if _real_command(masked, m.start())]
    if ends:
        result["body"] = expanded[begin.end():ends[0].start()]
        result["trailing_text"] = expanded[ends[0].end():]
    else:
        result["body"] = expanded[begin.end():]
        issue("missing_end_document", "No end{document}; retained body through EOF.")

    # Encoding fallback is informational; unresolved includes/control flow are not.
    if any(item["code"] != "encoding_fallback" for item in result["issues"]):
        result["status"] = "partial"
    return result

## 8. Chạy gather và đọc báo cáo

- `ok`: không phát hiện vấn đề ở các cấu trúc được hỗ trợ (không đảm bảo tương đương PDF).
- `partial`: có output nhưng còn include/control flow chưa xử lý chắc chắn; kiểm tra warnings.
- `no_document`: không thấy begin document sau gom; chưa có body sử dụng được.
- `error`: lỗi đọc root hoặc vượt giới hạn; không sử dụng output để dựng cây.

Đây là trạng thái nội bộ của gather, không tự ghi đè `parse_status` cuối cùng của Member 1.

In [ ]:
gather_result = None

if root_file is None:
    print("Không có root hợp lệ. Kiểm tra Task 1 trước.")
else:
    gather_result = gather(source_dir, root_file)
    print("Gather status:", gather_result["status"])
    print("Root:", gather_result["root_path"])
    print("Document found:", gather_result["document_found"])
    print("Unique files read:", len(gather_result["files_read"]))
    print("Include occurrences:", len(gather_result["include_events"]))
    print("Expanded characters:", gather_result["expanded_char_count"])
    print("Preamble characters:", len(gather_result["preamble"]))
    print("Body characters:", len(gather_result["body"]))
    print("\nFiles read:")
    for file in gather_result["files_read"]:
        print("-", file)
    print("\nWarnings:")
    if not gather_result["warnings"]:
        print("(none)")
    for warning in gather_result["warnings"]:
        print("-", warning)

    events = gather_result["include_events"]
    if events:
        try:
            import pandas as pd
            from IPython.display import display
        except ImportError:
            for event in events[:30]:
                print(event)
        else:
            display(pd.DataFrame(events).head(30))

## 9. Xem nội dung và chuyển cho bước sau

Preamble giữ định nghĩa macro cho Member 4. Body giữ LaTeX và citation để chuẩn hóa
và dựng cây sau này; chưa xóa bibliography tại Task 2 vì Member 3 còn cần trích xuất.
`expanded_text` giữ toàn bộ source sau gom; `trailing_text` là phần sau end document.
Không có file nào được ghi lên Drive khi chạy các cell này.

In [ ]:
preamble = None
document_body = None

if gather_result is not None and gather_result["document_found"]:
    print("--- PREAMBLE: tối đa 1500 ký tự ---")
    print(gather_result["preamble"][:1500])
    print("\n--- BODY: tối đa 4000 ký tự ---")
    print(gather_result["body"][:4000])

    if gather_result["status"] == "ok":
        preamble = gather_result["preamble"]
        document_body = gather_result["body"]
        print("\nĐã đặt preamble và document_body cho bước tiếp theo.")
    else:
        print("\nOutput chưa đầy đủ: kiểm tra warnings trước khi chuyển sang dựng cây.")
else:
    print("Chưa có document body. Xem kết quả Task 1 và báo cáo gather.")

## Kiểm tra với dữ liệu nhóm

1. Chạy paper `2609-37833/v1`: nếu chỉ có main.tex và không có input/include, số lần include bằng 0 là hợp lệ.
2. Thử version có nhiều file được input/include: xem target và thứ tự các event.
3. Kiểm tra body không chứa preamble, còn các section, công thức, figure và bibliography cần xử lý tiếp.
4. Chọn một file .tex không được include trong version nếu có; xác nhận file đó không nằm trong files_read.
5. Gửi Gather status, số file/lần include và warnings để kiểm tra các trường hợp thật.

Khi đổi paper/version, chạy lại từ cell cấu hình, Task 1 rồi Task 2 để tránh dùng biến cũ.
Trước khi commit, xóa output dài/đường dẫn cá nhân; không commit dữ liệu M1.